# Module 19 — The Standard Library, Files, and Serialization

## Exercise 19.3 — Twelve datetime puzzles

Predict each answer before running. Every one has a timezone or DST trap.
Run:  python ex03_datetime.py

---

**How to work through this.** Each task below is its own cell. Run them one at a
time and read the output before moving on; that is the whole advantage of a
notebook over a script. Where a cell asks for a prediction, write it before you
run anything. Being wrong on purpose in a place where it costs nothing is how
the correct model gets built.

---

# The concepts behind this exercise

Read this before the tasks. Every idea the tasks below use is explained here, so
you should not need to leave this notebook.

The code cells in this part are demonstrations rather than exercises. Run them,
change a value, run them again. That is the whole point of having them here
instead of in a document.

## Concept 2. Files and I/O

In [ ]:
with path.open("r", encoding="utf-8", newline="") as fh: ...
with path.open("rb") as fh: ...             # binary: no encoding, no newline

| Mode | Meaning |
|---|---|
| `r` `w` `a` | read, truncate-and-write, append |
| `x` | create, fail if it exists — the safe way to avoid clobbering |
| `+` | read *and* write |
| `b` | binary |

Three things worth knowing:

**`newline=""` for the `csv` module.** Without it, `\r\n` inside a quoted field
is translated and the file is corrupted. The `csv` docs say this and everyone
skips it.

**Iterating a file yields lines lazily.** `for line in fh:` reads a buffer at a
time, so it works on a file larger than memory. `fh.readlines()` does not.

**Text mode does newline translation and encoding.** Binary mode does neither.
If you are computing a hash, comparing bytes, or handling anything non-text, use
binary.

In [ ]:
import shutil, tempfile, os

shutil.copy2(src, dst)              # copies metadata too
shutil.move(src, dst)
shutil.rmtree(path)
shutil.disk_usage(path)

with tempfile.TemporaryDirectory() as td: ...       # cleaned up always
with tempfile.NamedTemporaryFile(delete=False) as f: ...

---

## Concept 3. `json`

In [ ]:
import json

json.dumps(obj, indent=2, sort_keys=True, ensure_ascii=False, default=str)
json.loads(text)
json.dump(obj, fh)      # to a file
json.load(fh)

**What JSON cannot represent, and what Python does about it:**

| Python | JSON | Round trip? |
|---|---|---|
| `dict` with str keys | object | yes |
| `dict` with int keys | object with **str** keys | **no** — `{1: "a"}` comes back `{"1": "a"}` |
| `tuple` | array | **no** — comes back as a `list` |
| `set`, `bytes`, `Decimal`, `datetime` | — | **no** — `TypeError` unless handled |
| `float('nan')`, `inf` | not valid JSON | Python emits `NaN` anyway, which other parsers reject |
| large `int` | number | yes, but many parsers lose precision above 2⁵³ |

In [ ]:
class Encoder(json.JSONEncoder):
    def default(self, obj: Any) -> Any:
        if isinstance(obj, datetime):
            return obj.isoformat()
        if isinstance(obj, Decimal):
            return str(obj)             # str, NOT float -- Module 03
        if isinstance(obj, set):
            return sorted(obj)
        return super().default(obj)

**`ensure_ascii=False`** keeps non-ASCII readable (`"café"` rather than
`"café"`) and produces smaller output. **`sort_keys=True`** makes output
deterministic, which matters for diffs, caching and content hashing.

---

## Concept 5. `sqlite3`

A full SQL database, in the standard library, with no server.

In [ ]:
import sqlite3

with sqlite3.connect("app.db") as conn:      # NOTE: commits, does NOT close
    conn.row_factory = sqlite3.Row           # dict-like rows
    conn.execute("PRAGMA foreign_keys = ON") # OFF by default!
    cur = conn.execute("SELECT * FROM users WHERE age > ?", (18,))
    for row in cur:
        print(row["name"])

**Always use parameters, never string formatting.**

In [ ]:
conn.execute(f"SELECT * FROM users WHERE name = '{name}'")   # SQL INJECTION
conn.execute("SELECT * FROM users WHERE name = ?", (name,))  # correct

This is not a style preference. `name = "'; DROP TABLE users; --"` is the entire
attack, and parameterisation makes it structurally impossible because the value
never becomes part of the statement.

Two sqlite-specific traps: `with conn:` is a **transaction** context manager,
not a closing one — it commits or rolls back and leaves the connection open.
And foreign key enforcement is **off** by default, so your constraints do
nothing until you turn the pragma on.

---

## Concept 7. `datetime`, and the one rule

In [ ]:
from datetime import datetime, date, timedelta, timezone, UTC
from zoneinfo import ZoneInfo          # 3.9+, real tz database

datetime.now(UTC)                       # aware. Correct.
datetime.now()                          # NAIVE. Almost always a bug.
datetime.now(ZoneInfo("Europe/London")) # aware, with DST handled

**Store and compute in UTC; convert to local time only for display.** A naive
datetime has no timezone, so comparing two of them from different sources is
meaningless, and arithmetic across a DST boundary is wrong.

In [ ]:
dt.isoformat()                          # '2026-08-03T12:00:00+00:00'
datetime.fromisoformat(text)            # 3.11+ parses almost anything ISO
dt.astimezone(ZoneInfo("Asia/Tokyo"))
dt.timestamp()                          # seconds since the epoch, UTC

Two things people get wrong:

**`timedelta` arithmetic is exact, calendar arithmetic is not.** Adding
`timedelta(days=30)` is 30×86400 seconds, which is not "one month" and is not
even 30 days across a DST change. For calendar months use
`dateutil.relativedelta`.

**`time.time()` for measuring intervals is wrong.** It is wall-clock and can
jump backwards (NTP, DST, a manual change). Use `time.perf_counter()` for
durations and `time.monotonic()` for timeouts.

---

---

# Now the exercise

You have everything you need. Work top to bottom, and where a cell asks for a
prediction, write it before you run anything.

## The concepts this exercise uses

These are the numbered sections of [the module README](../README.md). If a task below stops making sense, the section named next to it is the one to re-read.

- Section 1: `pathlib`
- Section 2: Files and I/O
- Section 3: `json`
- Section 4: `csv`
- Section 5: `sqlite3`
- Section 6: `pickle`, and why not to use it
- Section 7: `datetime`, and the one rule
- Section 8: `re`, at working depth
- Section 9: `subprocess`, safely
- Section 10: The rest, in one table

> The teaching for this module currently lives in the README rather than in this notebook. Read it alongside these cells.

## Setup

Run this first. It is the imports and any shared values the tasks below need.

In [ ]:
from __future__ import annotations

import time
from datetime import date, datetime, timedelta, timezone
from zoneinfo import ZoneInfo

UTC = timezone.utc
LONDON = ZoneInfo("Europe/London")
NEW_YORK = ZoneInfo("America/New_York")
TOKYO = ZoneInfo("Asia/Tokyo")

---

## `q01`

_q01_

In [ ]:
def q01() -> None:
    # PREDICTION: can these be compared? What happens?
    naive = datetime(2026, 6, 1, 12, 0)
    aware = datetime(2026, 6, 1, 12, 0, tzinfo=UTC)
    try:
        print("q01", naive == aware, naive < aware)
    except TypeError as exc:
        print("q01 TypeError:", exc)

---

## `q02`

_q02_

In [ ]:
def q02() -> None:
    # PREDICTION: adding 24 hours across a DST boundary.
    # 2026-03-29 is when the UK springs forward.
    before = datetime(2026, 3, 29, 0, 30, tzinfo=LONDON)
    print("q02 +timedelta(hours=24):", before + timedelta(hours=24))
    print("q02 +timedelta(days=1):  ", before + timedelta(days=1))

---

## `q03`

_q03_

In [ ]:
def q03() -> None:
    # PREDICTION: this local time occurs TWICE in 2026. Which one is this?
    ambiguous = datetime(2026, 10, 25, 1, 30, tzinfo=LONDON)
    print("q03", ambiguous, ambiguous.utcoffset())
    print("q03 fold=1:", ambiguous.replace(fold=1),
          ambiguous.replace(fold=1).utcoffset())

---

## `q04`

_q04_

In [ ]:
def q04() -> None:
    # PREDICTION: this local time NEVER occurs in 2026. What does Python do?
    nonexistent = datetime(2026, 3, 29, 1, 30, tzinfo=LONDON)
    print("q04", nonexistent, nonexistent.utcoffset())
    print("q04 in UTC:", nonexistent.astimezone(UTC))

---

## `q05`

_q05_

In [ ]:
def q05() -> None:
    # PREDICTION: same instant, three renderings.
    instant = datetime(2026, 8, 3, 12, 0, tzinfo=UTC)
    for tz, label in [(LONDON, "London"), (NEW_YORK, "New York"), (TOKYO, "Tokyo")]:
        print(f"q05 {label:<10}", instant.astimezone(tz))

---

## `q06`

_q06_

In [ ]:
def q06() -> None:
    # PREDICTION: is this the right way to get "now" in a timezone?
    print("q06 a:", datetime.now(TOKYO))
    print("q06 b:", datetime.now().replace(tzinfo=TOKYO))
    print("q06 c:", datetime.now(UTC).astimezone(TOKYO))

---

## `q07`

_q07_

In [ ]:
def q07() -> None:
    # PREDICTION: what does subtracting two aware datetimes in DIFFERENT zones
    # give you?
    a = datetime(2026, 8, 3, 12, 0, tzinfo=LONDON)
    b = datetime(2026, 8, 3, 12, 0, tzinfo=TOKYO)
    print("q07", b - a, a == b)

---

## `q08`

_q08_

In [ ]:
def q08() -> None:
    # PREDICTION: round-tripping through isoformat.
    original = datetime(2026, 8, 3, 12, 0, 30, 123456, tzinfo=TOKYO)
    text = original.isoformat()
    back = datetime.fromisoformat(text)
    print("q08", text)
    print("q08 equal?", back == original, "| same tzinfo?", back.tzinfo == original.tzinfo)

---

## `q09`

_q09_

In [ ]:
def q09() -> None:
    # PREDICTION: date arithmetic on month boundaries.
    d = date(2026, 1, 31)
    print("q09 +30 days:", d + timedelta(days=30))
    try:
        print("q09 +1 month:", d.replace(month=2))
    except ValueError as exc:
        print("q09 replace(month=2):", exc)

---

## `q10`

_q10_

In [ ]:
def q10() -> None:
    # PREDICTION: which of these can go backwards?
    t1, p1, m1 = time.time(), time.perf_counter(), time.monotonic()
    time.sleep(0.01)
    t2, p2, m2 = time.time(), time.perf_counter(), time.monotonic()
    print(f"q10 time():        {t2 - t1:.6f}")
    print(f"q10 perf_counter():{p2 - p1:.6f}")
    print(f"q10 monotonic():   {m2 - m1:.6f}")

---

## `q11`

_q11_

In [ ]:
def q11() -> None:
    # PREDICTION: timestamps and naive datetimes.
    ts = 1785000000
    print("q11 utcfromtimestamp:", datetime.fromtimestamp(ts, UTC))
    print("q11 fromtimestamp   :", datetime.fromtimestamp(ts))

---

## `q12`

_q12_

In [ ]:
def q12() -> None:
    # PREDICTION: storing a future appointment.
    # A user in London books a meeting for 2026-10-25 at 01:30 local time.
    # You store it as UTC. The DST rules for 2027 then change (governments do
    # this). What is wrong with what you stored?
    print("q12 -- no code. Answer in a comment:")
    print("     for a FUTURE local appointment, should you store UTC or")
    print("     local-time-plus-zone-name? Justify. Which does Google Calendar")
    print("     do, and why?")

---

## Run it

This is what running the original file did. Everything above must have been run first.

In [ ]:
if __name__ == "__main__":
    for fn in [q01, q02, q03, q04, q05, q06, q07, q08, q09, q10, q11, q12]:
        fn()

---

## Before you move on

- [ ] Every cell above ran, in order, on a fresh kernel.
- [ ] You wrote a prediction before running, wherever one was asked for.
- [ ] You can say in one sentence what each task was actually testing.
- [ ] Anything that surprised you is written down in `PROGRESS.md`.

Compare against the worked answers in `../solutions/` only after your own
attempt runs.